# Figure 6: Integrated immune phenotypes and divergent aging trajectories

This notebook reproduces panels of **Figure 6** of the AIDA AIRR manuscript:

- **Fig. 6A** — Correlation matrix of standardized immune features across donors
- **Fig. 6B** — Heat map characterising K-means immune clusters by demographic composition
- **Fig. 6C** — Force-directed UMAP of integrated immune landscape, colored by cluster and demographics
- **Fig. 6D** — Force-directed UMAP colored by major immune subset proportions
- **Fig. 6E** — Circular bar plots summarising the feature profiles of each immune phenotype


## Imports

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

import numpy as np
import pandas as pd
import scipy as sp
import scipy.stats as stats
from scipy.cluster.hierarchy import linkage, leaves_list
from math import pi

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import gridspec
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns

import scanpy as sc
import anndata as ad
import sceleto2 as scjp

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

%matplotlib inline
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, color_map='OrRd')

plt.rcParams['pdf.fonttype'] = 42
sns.set_style('ticks', {"grid.color": "dimgray", "grid.linestyle": ":",
                         'axes.edgecolor': 'black', 'axes.edgewidth': 2})
sns.set_context("paper", font_scale=1.3, rc={'patch.linewidth': 1})

Ethnicity_order = ['Chinese', 'Malay', 'Indian', 'Japanese', 'Korean', 'Thai']
Ethnicity_colors = ["tomato", 'gold', "dodgerblue", "limegreen", "aquamarine", "orchid", 'gray']


## Load all source AnnData (T/NK, B, Myeloid, V(D)J)

In [ ]:
tdata = sc.read('data/06_250529_TDATA_anno2_meta.h5ad')

In [ ]:
bdata = sc.read('data/06_250608_BDATA_IGLK_annotated_meta.h5ad')

In [ ]:
mdata = sc.read('data/05_240712_MDATA_annotated_meta.h5ad')

## QC: filter low-cell-count donors and intersect across modalities

In [ ]:
tdata = tdata[tdata.obs['Ethnicity']!='European']
tdata = tdata[tdata.obs['Country']!='IN']
tdata = tdata[~tdata.obs['Ethnicity'].isna()]

bdata = bdata[bdata.obs['Ethnicity']!='European']
bdata = bdata[~bdata.obs['Ethnicity'].isna()]
bdata = bdata[bdata.obs['Country']!='IN']

mdata = mdata[mdata.obs['Ethnicity']!='European']
mdata = mdata[~mdata.obs['Ethnicity'].isna()]
mdata = mdata[mdata.obs['Country']!='IN']

Ethnicity_order=['Chinese', 'Malay','Indian','Japanese','Korean','Thai']

bdata.obs['PatientID'] = bdata.obs['PatientID'].astype('str')

tdata.obs['PatientID'] = tdata.obs['PatientID'].astype('str')

mdata.obs['PatientID'] = mdata.obs['PatientID'].astype('str')

In [ ]:
counts = bdata.obs['PatientID'].value_counts()

# 2) 90th percentile (상위 10%) 임계값 계산
threshold = counts.quantile(0.1)

# 3) 분위수 이상인 환자 리스트 생성
ptl = counts[counts >= threshold].index.tolist()

In [ ]:
ptl = counts[counts >= np.percentile(counts, 5)].index.tolist()


In [ ]:
bdata = bdata[bdata.obs['PatientID'].isin(ptl)]

In [ ]:
counts = tdata.obs['PatientID'].value_counts()

# 2) 90th percentile (상위 10%) 임계값 계산
threshold = counts.quantile(0.1)

# 3) 분위수 이상인 환자 리스트 생성
ptl = counts[counts >= threshold].index.tolist()

In [ ]:
ptl = counts[counts >= np.percentile(counts, 5)].index.tolist()


In [ ]:
tdata = tdata[tdata.obs['PatientID'].isin(ptl)] 

In [ ]:
counts = mdata.obs['PatientID'].value_counts()

# 2) 90th percentile (상위 10%) 임계값 계산
threshold = counts.quantile(0.1)

# 3) 분위수 이상인 환자 리스트 생성
ptl = counts[counts >= threshold].index.tolist()

In [ ]:
ptl = counts[counts >= np.percentile(counts, 5)].index.tolist()


In [ ]:
mdata = mdata[mdata.obs['PatientID'].isin(ptl)]

In [ ]:
vdata = sc.read('data/06_250601_TDATA_scirpy_merging_meta_with_clone_id_Pt.h5ad')

vdata = vdata[vdata.obs['Ethnicity']!='European']
vdata = vdata[~vdata.obs['Ethnicity'].isna()]
vdata = vdata[vdata.obs['Country']!='IN']

In [ ]:
counts = vdata.obs['PatientID'].value_counts()

# 2) 90th percentile (상위 10%) 임계값 계산
threshold = counts.quantile(0.1)

# 3) 분위수 이상인 환자 리스트 생성
ptl = counts[counts >= threshold].index.tolist()

print('original {}'.format(len(vdata.obs['PatientID'].unique())))
print('after {}'.format(len(ptl)))

In [ ]:
vdata = vdata[vdata.obs['PatientID'].isin(ptl)]

Per-cell Gini coefficient (re-used from Figure 2 pipeline):

In [ ]:
def gini_index(array):
    """
    주어진 배열에 대한 Gini 지수를 계산합니다.
    :param array: 수입이나 자산 분포를 나타내는 숫자 배열
    :return: Gini 지수 (0에서 1 사이의 값)
    """
    # 배열 sort
    array = np.sort(array)
    index = np.arange(1, len(array) + 1)
    n = len(array)
    
    ## TCR Gini coefficient 지수 계산
    return ((2 * np.sum(index * array)) / (n * np.sum(array))) - ((n + 1) / n)

In [ ]:
bdata.obs['clone_id_Pt'] = ['{}*{}'.format(a,b) for a, b in zip(bdata.obs['clone_id'],
                                                                bdata.obs['PatientID'])]

In [ ]:
bdata.obs['Pt_anno1'] = ['{}*{}'.format(a,b) for a,b in zip(bdata.obs['PatientID'],bdata.obs['anno1'])]

In [ ]:
bdata.obs['Pt_anno1'] = bdata.obs['Pt_anno1'].astype('str')
bdata.obs['clone_id_Pt'] = bdata.obs['clone_id_Pt'].astype('str')

gindex = {}
for a in bdata.obs['Pt_anno1'].unique():
    gindex[a] = gini_index(bdata.obs[bdata.obs['Pt_anno1']==a]['clone_id_Pt'].value_counts().values)

gdf = pd.DataFrame.from_dict(gindex, orient='index')
gdf = gdf.reset_index()
gdf.columns = ['Pt_anno1','gini_anno1']

test = bdata.obs.merge(gdf, on='Pt_anno1', how='left')
test.index = bdata.obs.index.copy()

bdata.obs = test.copy()

In [ ]:
vdata.obs['anno2'] = np.where(vdata.obs['anno2'].str.startswith('T_CD4_N'), 'T_CD4_Naive', vdata.obs['anno2'])
vdata.obs['anno2'] = np.where(vdata.obs['anno2'].str.startswith('T_CD8_N'), 'T_CD8_Naive', vdata.obs['anno2'])

In [ ]:
vdata.obs['Pt_anno2'] = ['{}*{}'.format(a,b) for a,b in zip(vdata.obs['PatientID'],vdata.obs['anno2'])]

In [ ]:
vdata.obs['Pt_anno1'] = ['{}*{}'.format(a,b) for a,b in zip(vdata.obs['PatientID'],vdata.obs['anno1'])]

In [ ]:
vdata.obs['Pt_anno2'] = vdata.obs['Pt_anno2'].astype('str')
vdata.obs['clone_id_Pt'] = vdata.obs['clone_id_Pt'].astype('str')

gindex = {}
for a in vdata.obs['Pt_anno2'].unique():
    gindex[a] = gini_index(vdata.obs[vdata.obs['Pt_anno2']==a]['clone_id_Pt'].value_counts().values)

gdf = pd.DataFrame.from_dict(gindex, orient='index')
gdf = gdf.reset_index()
gdf.columns = ['Pt_anno2','gini_anno2']

test = vdata.obs.merge(gdf, on='Pt_anno2', how='left')
test.index = vdata.obs.index.copy()

vdata.obs = test.copy()

In [ ]:
vdata.obs['Pt_anno1'] = vdata.obs['Pt_anno1'].astype('str')
vdata.obs['clone_id_Pt'] = vdata.obs['clone_id_Pt'].astype('str')

gindex = {}
for a in vdata.obs['Pt_anno1'].unique():
    gindex[a] = gini_index(vdata.obs[vdata.obs['Pt_anno1']==a]['clone_id_Pt'].value_counts().values)

gdf = pd.DataFrame.from_dict(gindex, orient='index')
gdf = gdf.reset_index()
gdf.columns = ['Pt_anno1','gini_anno1']

test = vdata.obs.merge(gdf, on='Pt_anno1', how='left')
test.index = vdata.obs.index.copy()

vdata.obs = test.copy()

### Helper: compute pairwise correlation + p-values

In [ ]:
def get_correlations(df):
    df = df.dropna()._get_numeric_data()
    dfcols = pd.DataFrame(columns=df.columns)
    pvalues = dfcols.transpose().join(dfcols, how="outer")
    correlations = dfcols.transpose().join(dfcols, how="outer")
    
    # 모든 p-value를 저장할 리스트
    all_pvalues = []
    
    for ix, r in enumerate(df.columns):
        for jx, c in enumerate(df.columns):
            sp = stats.pearsonr(df[r], df[c])
            correlations[c][r] = sp[0]
            pvalues[c][r] = sp[1]
            if ix > jx:  # 대각선 아래 값만 저장 (중복 제거)
                all_pvalues.append(sp[1])
    
    # Benjamini-Hochberg 방법을 사용하여 p-value 교정
    _, corrected_pvalues, _, _ = multipletests(all_pvalues, method='fdr_bh')
    
    # 교정된 p-value를 저장할 새로운 DataFrame 생성
    corrected_pvalues_df = dfcols.transpose().join(dfcols, how="outer")
    
    # 교정된 p-value를 DataFrame에 채우기
    corrected_pvalue_idx = 0
    for ix, r in enumerate(df.columns):
        for jx, c in enumerate(df.columns):
            if ix > jx:
                corrected_pvalues_df[c][r] = corrected_pvalues[corrected_pvalue_idx]
                corrected_pvalues_df[r][c] = corrected_pvalues[corrected_pvalue_idx]  # 대칭적으로 채우기
                corrected_pvalue_idx += 1
            elif ix == jx:
                corrected_pvalues_df[c][r] = 1.0  # 대각선은 1로 설정
    
    return (correlations.astype("float"), 
            pvalues.astype("float"),
            corrected_pvalues_df.astype("float"))

In [ ]:
vdata = vdata[~vdata.obs['IR_VDJ_1_c_call'].isna()]
vdata = vdata[~vdata.obs['IR_VJ_1_c_call'].isna()] 

In [ ]:
idata = bdata[~bdata.obs['v_call_B_VDJ'].str.startswith('N')]

In [ ]:
idata = idata[~idata.obs['c_call_VDJ'].str.startswith('N')]
idata = idata[~idata.obs['c_call_VDJ'].str.startswith('IGL')]
idata = idata[~idata.obs['c_call_VDJ'].str.startswith('IGK')]

In [ ]:
idata.obs['PatientID'] = idata.obs['PatientID'].astype('str')

In [ ]:
mdata.obs['PatientID'] = mdata.obs['PatientID'].astype('str')

In [ ]:
bdata.obs['PatientID'] = bdata.obs['PatientID'].astype('str')

In [ ]:
vdata.obs['PatientID'] = vdata.obs['PatientID'].astype('str')

In [ ]:
tdata.obs['PatientID'] = tdata.obs['PatientID'].astype('str')

In [ ]:
mata = tdata.obs.drop_duplicates("PatientID")[['PatientID','Country', 'BMI', 'Age', 'Sex',
       'Ethnicity']]

In [ ]:
tdata.obs['anno2'] = np.where(tdata.obs['anno2'].str.startswith('T_CD4_N'), 'T_CD4_Naive', tdata.obs['anno2'])
tdata.obs['anno2'] = np.where(tdata.obs['anno2'].str.startswith('T_CD8_N'), 'T_CD8_Naive', tdata.obs['anno2'])

### Donors with paired T, B, and myeloid annotations

In [ ]:
interptl = set(mdata.obs['PatientID']) & set(bdata.obs['PatientID']) & set(tdata.obs['PatientID']) & set(vdata.obs['PatientID']) & set(idata.obs['PatientID'])

In [ ]:
bdata = bdata[bdata.obs['PatientID'].isin(interptl)]

In [ ]:
tdata = tdata[tdata.obs['PatientID'].isin(interptl)]
mdata = mdata[mdata.obs['PatientID'].isin(interptl)]

In [ ]:
vdata = vdata[vdata.obs['PatientID'].isin(interptl)]

In [ ]:
idata = idata[idata.obs['PatientID'].isin(interptl)]

In [ ]:
mata = mata[mata['PatientID'].isin(interptl)]

## Build per-donor feature matrix

For each donor we collect:

- T cell subset proportions (`sf`)
- B cell subset proportions (`bf`)
- Myeloid subset proportions (`myf`)
- Memory B cell isotype proportions (`isof`)
- Memory B cell average SHM (`mf`)
- Per-subset TCR Gini coefficient (`gmf`)
- Memory B cell BCR Gini coefficient (`gbf`)


In [ ]:
# Per-donor T cell subset proportion (`sf`)
sf = pd.crosstab(tdata.obs['PatientID'], tdata.obs['anno2'], normalize=0)
sf.columns = [c for c in sf.columns]

# B cell subset proportion (`bf`)
bf = pd.crosstab(bdata.obs['PatientID'], bdata.obs['anno2'], normalize=0)

# Myeloid subset proportion (`myf`)
myf = pd.crosstab(mdata.obs['PatientID'], mdata.obs['anno2'], normalize=0)
myf.columns = ['M_' + c if not c.startswith('M_') else c for c in myf.columns]

# Memory B cell isotype proportions (`isof`)
mbc = bdata[bdata.obs['anno1'] == 'B_Memory']
isof = pd.crosstab(mbc.obs['PatientID'], mbc.obs['c_call_VDJ'], normalize=0)

# Memory B cell average SHM rate (`mf`) -- mu_freq
mf = mbc.obs.groupby('PatientID').mean()[['mu_freq']]


In [ ]:
gmf = vdata.obs.pivot_table(values='gini_anno2', index='PatientID', columns='anno2', aggfunc='mean')
gmf.columns = [f'{col}-gini' for col in gmf.columns]

In [ ]:
gbf = bdata.obs.pivot_table(values='gini_anno1', index='PatientID', columns='anno1', aggfunc='mean')
gbf.columns = [f'{col}-gini' for col in gbf.columns]

In [ ]:
gbf = gbf[['B_Memory-gini']]

In [ ]:
gmf = gmf[['T_CD4_CTL-gini', 
       'T_CD4_Th1-gini', 'T_CD4_Th17-gini', 'T_CD4_Th2-gini',
       'T_CD4_Treg-gini', 'T_CD4_activated-gini', 'T_CD4_cTfh-gini',
       'T_CD8_KIR-gini', 'T_CD8_TEM_GZMB-gini',
       'T_CD8_TEM_GZMK-gini', 'T_unc_MAIT-gini']]

In [ ]:

# isof =  isof[isof.index.isin(sf.index)]
# mf =  mf[mf.index.isin(sf.index)]
metz = mata.set_index('PatientID')[['Ethnicity','Sex','Age','BMI']]
# metz = mata.set_index('PatientID')[['Age']]
# metz =  metz[metz.index.isin(sf.index)]
datas = pd.concat([sf,bf,isof,mf,myf, gmf, gbf], axis=1)
edatas = pd.concat([sf,bf,isof,mf,myf,gmf, gbf, metz], axis=1)
# datas = pd.concat([sf,bf, myf], axis=1)
# edatas = pd.concat([sf,bf,myf, metz], axis=1)


In [ ]:
datas = datas.fillna(0)

In [ ]:
scaler = StandardScaler()
datas_scaled = scaler.fit_transform(datas)

In [ ]:
datas_scaled = pd.DataFrame(scaler.fit_transform(datas), index=datas.index, columns=datas.columns)


## Fig. 6A — Correlation matrix of immune features

In [ ]:
origin = pd.Series(index=datas_scaled.columns)
origin[sf.columns] = 'T'
origin[bf.columns] = 'B'
origin[myf.columns] = 'M'
origin[isof.columns] = 'Isotype' 
origin[mf.columns] = 'SHM'
origin[gmf.columns] = 'Clonality'
origin[gbf.columns] = 'Clonality'

lut = {'T': 'crimson', 'B': '#56B4E9','M':'#009E73','Isotype':'darkblue','SHM':'purple', 'Clonality':'chocolate'}
col_colors = origin.map(lut)

import matplotlib.colors as mcolors

col_colors_faded = [mcolors.to_rgba(c, alpha=0.2) for c in col_colors]

correlations, uncorrected_p_values, corrected_p_values = get_correlations(datas_scaled)

# Correct p-values for multiple testing and check significance (True if the corrected p-value < 0.05)
shape = corrected_p_values.values.shape
significant_matrix = multipletests(uncorrected_p_values.values.flatten())[0].reshape(
    shape
)

# Here we start plotting
g = sns.clustermap(datas_scaled.corr(), cmap="vlag", vmin=-1, vmax=1, figsize=(21,21),
               linewidths=0.75, linecolor='lightgray', 
                    col_colors=col_colors_faded, row_colors=col_colors_faded, 
                        cbar_pos=(0.1, .1, .01, .2),)


clustered_corr = g.data2d
clustered_sig  = corrected_p_values.loc[clustered_corr.index,
                                        clustered_corr.columns]

clustered_sig = clustered_sig<0.05

ya = []
for label in g.ax_heatmap.get_yticklabels():
    ya.append(label.get_text())

xa = []
for label in g.ax_heatmap.get_xticklabels():
    xa.append(label.get_text())

#corp = pd.edataFrame(multipletests(uncorrected_p_values.values.flatten())[0].reshape((35,35)))

#corp.columns = uncorrected_p_values.columns

#corp.index = uncorrected_p_values.index
corp = corrected_p_values<0.05

corp = corp[xa]

corp = corp.T[ya].T

for i, row in enumerate(clustered_corr.index):
    for j, col in enumerate(clustered_corr.columns):
        if row != col and clustered_sig.loc[row, col]:
            g.ax_heatmap.text(j+0.5, i+0.5, "*",
                              ha="center", va="center")



 

# # Here we add asterisks onto cells with signficant correlations
# for i, ix in enumerate(corp.columns):
#     for j, jx in enumerate(corp.index):
#         if i != j:
#             text = g.ax_heatmap.text(
#                 j + 0.5,
#                 i + 0.6,
#                 "*" if corp.values[i, j] else "",
#                 ha="center",
#                 va="center",
#                 color="k",
#             )
#             text.set_fontsize(13)
            
            
for label in g.ax_heatmap.get_xticklabels():
    col_name = label.get_text()  # 현재 tick label의 텍스트 (컬럼 이름)
    if col_name in origin.index:
        # 해당 컬럼의 원본에 따라 글씨색 변경
        label.set_color(lut[origin[col_name]])
        
for label in g.ax_heatmap.get_yticklabels():
    col_name = label.get_text()  # 현재 tick label의 텍스트 (컬럼 이름)
    if col_name in origin.index:
        # 해당 컬럼의 원본에 따라 글씨색 변경
        label.set_color(lut[origin[col_name]])        

cbar = g.fig.get_children()[-1] 
cbar.spines[["bottom", "top", "left","right"]].set_visible(True)
cbar.spines[["bottom", "top", "left","right"]].set_color("k")

g.ax_row_dendrogram.remove()
g.ax_col_dendrogram.remove()

# plt.savefig('Fig7_clustermap.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight') 


## K-means clustering of donors

In [ ]:
expression_data = datas.dropna()
# expression_data = edf.iloc[:,:-1] 

In [ ]:
edatas = edatas[edatas.index.isin(expression_data.index)]

In [ ]:
gene_names = expression_data.columns
sample_names = expression_data.index
group_labels = edatas['Ethnicity']

### Elbow method to choose k (k=5 used in main figure)

In [ ]:
k_range = range(1, 11)
inertias = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
    kmeans.fit(expression_data_scaled)
    inertias.append(kmeans.inertia_)

ks = np.array(list(k_range))
inertias = np.array(inertias)

# ----------------------------
# 2) elbow point 계산
#    방법: 첫점-끝점 직선에서의 최대 거리
# ----------------------------
# 점들 (k, inertia)
points = np.column_stack((ks, inertias))

# 시작점, 끝점
start = points[0]
end = points[-1]

# 직선 벡터
line_vec = end - start
line_vec_norm = line_vec / np.linalg.norm(line_vec)

# 각 점에서 start까지의 벡터
vec_from_start = points - start

# 각 점의 직선 위 projection
proj_lengths = np.dot(vec_from_start, line_vec_norm)
proj_points = np.outer(proj_lengths, line_vec_norm) + start

# 점과 직선 사이 거리
distances = np.linalg.norm(points - proj_points, axis=1)

# elbow index / k
elbow_idx = np.argmax(distances)
elbow_k = ks[elbow_idx]
elbow_inertia = inertias[elbow_idx]

print(f"Estimated elbow point: k = {elbow_k}")
print(f"Inertia at elbow: {elbow_inertia:.2f}")

# ----------------------------
# 3) plot
# ----------------------------
plt.figure(figsize=(7,5))
plt.plot(ks, inertias, marker='o', label='Inertia')
plt.scatter(elbow_k, elbow_inertia, s=120, marker='*', label=f'Elbow k={elbow_k}')
plt.axvline(elbow_k, linestyle='--', alpha=0.7)

plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow plot with estimated elbow point")
plt.xticks(ks)
plt.legend()
plt.tight_layout()
plt.show()

### Run final K-means with k=6 (the elbow)

In [ ]:
n_clusters = 6
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
kmeans.fit(expression_data_scaled)
cluster_labels = kmeans.labels_
print(f'KMeans inertia: {kmeans.inertia_:.2f}')
print(f'Silhouette score: {silhouette_score(expression_data_scaled, cluster_labels):.3f}')


In [ ]:
edatas['cluster'] = cluster_labels

## Fig. 6B — Demographic composition heatmap per K-means cluster

In [ ]:
ethnicity_cluster_crosstab2 = pd.crosstab(edatas['cluster'], edatas['Ethnicity'], normalize=1)

In [ ]:
#  

# 데이터 준비
ethnicity_cols = ['Chinese', 'Malay', 'Indian', 'Japanese', 'Korean', 'Thai']
ethnicity_heatmap = ethnicity_cluster_crosstab2[['Chinese', 'Malay', 'Indian', 'Japanese', 'Korean', 'Thai']]
age_heatmap = edatas.groupby('cluster').median()[['Age']].round(1)
bmi_heatmap = edatas.groupby('cluster').median()[['BMI']].round(1)
male_heatmap = pd.crosstab(edatas['cluster'], edatas['Sex'], normalize=0)[['Male']].round(2)

# Create figure with custom grid
fig = plt.figure(figsize=(10, 4))
gs = gridspec.GridSpec(1, 4, width_ratios=[6, 1, 1, 1])

# 1. Ethnicity proportion heatmap
ax0 = plt.subplot(gs[0])
sns.heatmap(ethnicity_heatmap, 
            annot=True, 
            fmt='.2f',
            cmap='Blues',
            cbar_kws={'label': 'Proportion'},
            ax=ax0,   linewidths=0.5, linecolor='gray')
plt.title('Ethnicity Distribution per Cluster', pad=10)
plt.xlabel('')
plt.ylabel('Cluster')

# 2. Age heatmap
ax1 = plt.subplot(gs[1])
sns.heatmap(age_heatmap, 
            annot=True, 
            fmt='.1f',
            cmap='YlOrRd',
            cbar_kws={'label': 'Years'},
            ax=ax1,
            yticklabels=[],
             linewidths=0.5, linecolor='gray')# Remove y-axis labels for shared axis
plt.title('Age', pad=10)
plt.xlabel('')

# 3. Male proportion heatmap
ax2 = plt.subplot(gs[2])
sns.heatmap(male_heatmap, 
            annot=True, 
            fmt='.2f',
            cmap='Greens',
            cbar_kws={'label': 'Proportion'},
            ax=ax2,
            yticklabels=[],
               linewidths=0.5, linecolor='gray')# Remove y-axis labels for shared axis
plt.title('Male Ratio', pad=10)
plt.xlabel('')
 
ax3 = plt.subplot(gs[3])
sns.heatmap(bmi_heatmap, 
            annot=True, 
            fmt='.2f',
            cmap='Purples',
            cbar_kws={'label': 'Proportion'},
            ax=ax3,
            yticklabels=[],
               linewidths=0.5, linecolor='gray')# Remove y-axis labels for shared axis
plt.title('BMI', pad=10)
plt.xlabel('')

# Adjust layout
plt.tight_layout()
# plt.savefig('Fig7_kmeans_cluster_distribution.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight') 

plt.show()

## Build force-directed (ForceAtlas2) UMAP of donors

We use Scanpy's PAGA + draw_graph to obtain a 2-D layout that preserves both local
and global structure across donors. Donors are colored by demographic factors and
by major cell-type proportions for Fig. 6C, D.

In [ ]:
scaler = StandardScaler()
datas_scaled = scaler.fit_transform(datas)

In [ ]:
kdata = ad.AnnData(datas_scaled)

In [ ]:
kdata.obs = edatas.iloc[:,-5:]

In [ ]:
kdata.raw = kdata.copy()

### Feature selection: rank features by ANOVA F-score and random forest importance

In [ ]:
X = kdata.X
y = kdata.obs['cluster'].astype(int)

In [ ]:
# 1a) ANOVA F-score
selector_f = SelectKBest(score_func=f_classif, k=30)
selector_f.fit(X, y)
top_idx_f = selector_f.get_support(indices=True)

# 1b) Mutual Information
selector_mi = SelectKBest(score_func=mutual_info_classif, k=30)
selector_mi.fit(X, y)
top_idx_mi = selector_mi.get_support(indices=True)

# Extract feature names
feature_names = kdata.var_names  
top_features_f   = feature_names[top_idx_f]
top_features_mi  = feature_names[top_idx_mi]


In [ ]:
print("Top ANOVA features:", top_features_f)
print("Top MI features:   ", top_features_mi)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=0)
rf.fit(X, y)
importances = rf.feature_importances_

# Rank features
feat_imp = pd.Series(importances, index=feature_names)
top_rf = feat_imp.nlargest(30)
print("Top RF features:\n", top_rf)

In [ ]:
kdata.var['highly_variable'] = False

In [ ]:
kdata.var['highly_variable'] = np.where(kdata.var.index.isin(top_features_f), True,
                                        kdata.var['highly_variable'])

In [ ]:
sc.tl.pca(kdata, use_highly_variable=False)

In [ ]:
sc.pp.neighbors(kdata, use_rep='X_pca', n_pcs=10, n_neighbors=10,
        metric='l2')

In [ ]:
sc.tl.umap(kdata,  min_dist=0.5)

In [ ]:
kdata.obs['cluster'] = kdata.obs['cluster'].astype('category')

In [ ]:
sc.tl.paga(kdata, groups='cluster')

# 2) (Optional) Visualize the PAGA abstracted graph
sc.pl.paga(kdata, threshold=0.05, show=False)  


In [ ]:
sc.tl.draw_graph(
    kdata,
    layout='fa', 
    random_state=0
)

In [ ]:
ddd = kdata.copy()

In [ ]:
ddd.X = edatas[kdata.var.index].values.copy()

In [ ]:
kdata.raw = ddd.copy()

In [ ]:
kdata.obs['kluster'] = kdata.obs['cluster'].copy()

In [ ]:
kdata.uns['kluster_colors'] = ['#3479C7','#A5DE5F','#797550','#A7BBC1','#FFA300','#AD0001']

In [ ]:
kdata.obs['Ethnicity'] = kdata.obs['Ethnicity'].cat.reorder_categories(Ethnicity_order)

In [ ]:
kdata.uns["Ethnicity_colors"] = Ethnicity_colors

### Fig. 6C — Force-directed UMAP, colored by K-means cluster, sex, age, ethnicity

In [ ]:
sc.pl.draw_graph(
    kdata,
    color='kluster',
    edges=False,   ec='k', linewidth=0.3,
                save='Fig7_FA_plot_kluster.pdf')

In [ ]:
sc.pl.draw_graph(kdata, color='Sex', edges=False, ec='k', linewidth=0.3,
                 save='Fig6_FA_plot_sex.pdf')
sc.pl.draw_graph(kdata, color='Age', edges=False, cmap='YlOrRd', ec='k', linewidth=0.3,
                 save='Fig6_FA_plot_Age.pdf')


In [ ]:
sc.pl.draw_graph(
    kdata,
    color='Ethnicity',
    edges=False, palette=Ethnicity_colors, ec='k', linewidth=0.3,   
                save='Fig7_FA_plot_Ethnicity.pdf')

### Fig. 6D — Force-directed UMAP, colored by major immune subset proportions

In [ ]:
sc.pl.draw_graph(
    kdata,
    color= ['T_CD4_CTL','T_CD8_TEM_GZMB','T_CD8_KIR'], vmax=[0.091, 0.211, 0.141],
    edges=False, cmap='YlOrRd', ec='k', linewidth=0.3,
            save='Fig7_FA_plot_CD8.pdf')

In [ ]:
sc.pl.draw_graph(
    kdata,
    color= ['T_CD4_Th1','T_CD4_Th17','T_CD4_Treg'],  vmax=[0.101, 0.071,0.05],
    edges=False, cmap='YlOrRd', ec='k', linewidth=0.3,
            save='Fig7_FA_plot_CD4.pdf')

In [ ]:
sc.pl.draw_graph(
    kdata,
    color= ['B_Plasma','B_Memory_switched','B_Memory_unswitched'],  vmax=[0.15, 0.611,0.511],
    edges=False, cmap='YlOrRd', ec='k', linewidth=0.3,
            save='Fig7_FA_plot_B_cluster.pdf')

In [ ]:
sc.pl.draw_graph(
    kdata,
    color= ['B_Naive','T_CD4_Naive','T_CD8_Naive'],  vmax=[0.751, 0.501, 0.351],
    edges=False, cmap='YlOrRd', ec='k', linewidth=0.3,
                 save='Fig7_FA_plot_Naive.pdf')


In [ ]:
sc.pl.draw_graph(
    kdata,
    color= ['M_Mono_CD14_IFN','T_CD4_IFN','B_Naive_IFN'],  vmax=[0.601, 0.041,0.071],
    edges=False, cmap='YlOrRd', ec='k', linewidth=0.3,             
    save='Fig7_FA_plot_IFN.pdf')


In [ ]:
sc.pl.draw_graph(
    kdata,
    color= ['T_CD4_CTL','T_CD8_TEM_GZMB','T_CD4_Th2','T_CD4_Th17','B_Plasma','IGHG2',
            'IGHA2','mu_freq','B_Naive', 'T_CD4_Naive','T_CD8_Naive','B_Memory_switched'],
    edges=False, cmap='Reds', ec='k', linewidth=0.3,
)

## Fig. 6E — Circular bar plots of feature profiles per immune phenotype

We compute the per-cluster mean of each scaled feature and visualise via 0-centered ring
heatmaps. Each cluster gets one polar subplot; bars extend outward from a baseline ring
(`r0`) for positive Z-scores and inward for negative Z-scores. Genes with |Z| > 0.5 are
labelled in bold; |Z| < 0.5 features are shown in faint grey.

In [ ]:
expression_data = datas.dropna()
# expression_data = edf.iloc[:,:-1] 

In [ ]:
# Compute the correlation matrix
corr_matrix = expression_data.corr()

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list

# Perform hierarchical clustering
linked = linkage(corr_matrix, method='average')

# Get the order of the leaves (features) from the dendrogram
feature_order = leaves_list(linked)

In [ ]:
# Get the list of features in the new order
ordered_features = corr_matrix.columns[feature_order].tolist()

In [ ]:
all_genes_ordered = ordered_features

# Per-cluster mean of scaled features
mean_expression_per_cluster = (
    pd.concat([datas_scaled, edatas[['cluster']]], axis=1).dropna()
    .groupby('cluster')[all_genes_ordered].mean()
)

# Re-scale per-feature so that the ring heatmap has a comparable Z-score across clusters
scaler = StandardScaler()
scaled_mean_expression = pd.DataFrame(
    scaler.fit_transform(mean_expression_per_cluster[all_genes_ordered]),
    index=mean_expression_per_cluster.index,
    columns=all_genes_ordered
)


In [ ]:
genes      = all_genes_ordered
theta      = np.linspace(0, 2*np.pi, len(genes), endpoint=False)
threshold  = 0.5
n_clusters = len(scaled_mean_expression)

# 고정 반지름(모든 서브플롯 동일)
inner_r = 0.2
outer_r = 1.4
ring_th = outer_r - inner_r
r0      = inner_r + ring_th * 0.3   # 0 위치(원하는 proportion로 조절 가능)

# 전 클러스터 공통 스케일
vmin = float(np.nanmin(scaled_mean_expression.values))
vmax = float(np.nanmax(scaled_mean_expression.values))
# 양/음수 각각의 최대폭 (0 기준 대칭 scaling)
pos_scale = vmax if vmax > 0 else 1.0
neg_scale = -vmin if vmin < 0 else 1.0

# 컬러 매핑: 0 중심
norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
cmap = plt.get_cmap('coolwarm')

# 2×3 polar 그리드
n_rows, n_cols = 2, 3
fig, axes = plt.subplots(
    n_rows, n_cols,
    subplot_kw=dict(polar=True),
    figsize=(22, 18),
    tight_layout=True
)
axes = axes.flatten()

bar_width = 2*np.pi/len(genes)
offset_out = 0.03 * ring_th
offset_in  = 0.03 * ring_th

for ax, (cluster, row) in zip(axes, scaled_mean_expression.iterrows()):
    vals = row.values.astype(float)

    # 각 gene 별 bottom/height 계산: r0를 기준으로 양수는 바깥, 음수는 안쪽
    heights_pos = (np.clip(vals, 0, None) / pos_scale) * (outer_r - r0)
    heights_neg = (np.clip(-vals, 0, None) / neg_scale) * (r0 - inner_r)

    bottoms = np.where(vals >= 0, r0, r0 - heights_neg)
    heights = np.where(vals >= 0, heights_pos, heights_neg)

    colors = cmap(norm(vals))

    ax.bar(
        theta, heights,
        width=bar_width,
        bottom=bottoms,
        color=colors,
        edgecolor='k',
        linewidth=0.6,
        align='edge'   # 원하는 경우 center로 변경 가능
    )

    # 라벨 배치: 양수는 바깥쪽, 음수는 안쪽에
    for angle, v, btm, hgt, gene in zip(theta, vals, bottoms, heights, genes):
        # 글자 회전/sort 공통 옵션
        kw = dict(
            ha='center', va='bottom',
            rotation=angle*180/np.pi,
            rotation_mode='anchor'
        )
        if v >= threshold:
            text_r = btm + hgt + offset_out   # 양수: 막대 위 바깥쪽
            ax.text(angle, text_r, gene, fontsize=12, fontweight='bold', color='black', **kw)
        else:
            # 음수이거나(또는) 작게 양수인 경우 연한 회색으로
            if v < 0:
                text_r = btm - offset_in      # 음수: 막대 안쪽(중심쪽)
                kw_gray = dict(kw, va='top')  # 안쪽 배치 시 살짝 위상 변경
            else:
                text_r = btm + hgt + offset_out
                kw_gray = kw
            ax.text(angle, text_r, gene, fontsize=10, color='gray', alpha=0.8, **kw_gray)

    ax.set_xticks([]); ax.set_yticks([])
    ax.set_rlim(0, outer_r * 1.08)   # 모든 서브플롯 동일 r-limit
    ax.set_title(f"Cluster {cluster}", fontsize=12, pad=6)

# 남는 축 숨기기 (클러스터 수 < 6일 때)
for ax in axes[n_clusters:]:
    ax.set_visible(False)

# 컬러바
cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
mpl.colorbar.ColorbarBase(cax, cmap=cmap, norm=norm, label="Scaled expression")

plt.subplots_adjust(wspace=0.1, hspace=0.2)
plt.savefig(
    'figures/cluster_circular_plot.pdf',
    dpi=300, format='pdf', transparent=True, bbox_inches='tight'
)
plt.show()